# Phase 2B — Approved data cleaning and deterministic feature engineering

**Prediction scenario:** prioritize customers **before the current marketing campaign starts**. This supersedes Phase 2A's provisional immediately-before-a-call assumption.

Only approved operations are implemented: remove exact duplicate original rows (keep first), exclude five current-campaign features, give three coded status fields human-readable categorical labels, and replace the pdays sentinel with two deterministic recency features. No scaling, model-oriented encoding, statistical imputation, split, cross-validation, resampling, fitted transformation, outlier removal or model training occurs.

## Sources and execution

Reviewed `README.md`, `MSI5001 Project Bank Telemarketing .docx`, `01_eda.ipynb`, `eda_summary.md`, `02_preprocessing_decisions.md`, `02_feature_preprocessing_decision_table.md` and `dataset.csv`. Phase 1's missing-DOCX statement is historical; the document has since been read. The user's Phase 2B instructions explicitly approve duplicate deletion and the five exclusions, replacing earlier tentative recommendations. The README headline says 21 columns, but its enumerated list and actual CSV have 20; no absent feature is invented.

**Google Colab:** upload this notebook and the original `dataset.csv`, then Run all. The first code cell prompts for the CSV if it is not beside the notebook. Alternatively mount Drive and set `DATA_PATH` and `OUTPUT_DIR` below. Standard Colab provides pandas, NumPy and IPython. Runtime execution needs only the CSV; keep project documents alongside it for context. Outputs are written beside the CSV by default, or to `OUTPUT_DIR`; in Colab use the Files panel to download them. Only `02_cleaned_dataset.csv` and `02_data_cleaning_summary.md` are written by notebook execution. The original CSV is never overwritten. No network access or package installation is required.

This notebook expects the reviewed 41,188-row, 20-column snapshot with 12 exact duplicates. Guards stop on a changed schema, unexpected labels or invalid domains; they do not silently delete or correct additional observations. Observed ranges are described separately from verified domain rules.


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = None  # e.g. '/content/drive/MyDrive/bank-marketing-ml/dataset.csv'
OUTPUT_DIR = None  # defaults to the directory containing the input CSV
if DATA_PATH is None:
    if Path('dataset.csv').is_file():
        DATA_PATH = Path('dataset.csv')
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError('Set DATA_PATH to the original dataset.csv') from exc
        files.upload()
        if not Path('dataset.csv').is_file():
            raise FileNotFoundError('Upload a file named dataset.csv')
        DATA_PATH = Path('dataset.csv')
DATA_PATH = Path(DATA_PATH).resolve()
OUTPUT_DIR = DATA_PATH.parent if OUTPUT_DIR is None else Path(OUTPUT_DIR).resolve()
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
raw = pd.read_csv(DATA_PATH)
raw_snapshot = raw.copy(deep=True)
EXPECTED_COLUMNS = ['age','job','marital','education','default','housing','loan',
                    'contact','month','day_of_week','duration','campaign','pdays','previous',
                    'emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed','y']
assert raw.columns.tolist() == EXPECTED_COLUMNS, 'Unexpected schema: review before cleaning'
assert raw.shape == (41188, 20), 'Unexpected input dimensions: review the new snapshot'
# Stop before cleaning/export if target labels are unexpected; do not remove them.
assert raw.y.notna().all() and raw.y.isin([0, 1]).all(), (
    'Unexpected target labels: ' + repr(raw.y.value_counts(dropna=False).to_dict()))
print('Original dimensions:', raw.shape)
print('Source SHA-256:', source_sha256)
display(raw.head())
display(raw.dtypes.rename('raw_dtype').to_frame())


Original dimensions: (41188, 20)
Source SHA-256: 81fd5eb4b659dddbc704edd90376c32714c3f6a43b887cb686762903bc793f0d


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,0,0,0,telephone,may,mon,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0
1,57,services,married,high.school,-1,0,0,telephone,may,mon,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0
2,37,services,married,high.school,0,1,0,telephone,may,mon,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,basic.6y,0,0,0,telephone,may,mon,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0
4,56,services,married,high.school,0,0,1,telephone,may,mon,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0


,raw_dtype
age,int64
job,object
marital,object
education,object
default,int64
housing,int64
loan,int64
contact,object
month,object
day_of_week,object


## 1. Validate raw values before applying approved changes

Categorical domains come from the README. Numeric statuses `-1/0/1` are categorical unknown/no/yes. Negative employment variation and confidence values are legitimate observations, not generic missing-value codes. Nonnegative integral age/contact counts/recency and finite economic values are checked; no upper cap, clipping or IQR filter is used. Age 17–98, high contact counts and repeated economic indicators are preserved.

An unexpected null, blank, category or invalid numeric value causes a review stop; the code does not impute or remove the affected records. These checks do not establish availability at campaign start.


In [2]:
CATEGORICAL_DOMAINS = {
    'job': {'admin.','blue-collar','entrepreneur','housemaid','management','retired',
            'self-employed','services','student','technician','unemployed','unknown'},
    'marital': {'divorced','married','single','unknown'},
    'education': {'basic.4y','basic.6y','basic.9y','high.school','illiterate',
                  'professional.course','university.degree','unknown'},
    'default': {-1,0,1}, 'housing': {-1,0,1}, 'loan': {-1,0,1},
    'contact': {'cellular','telephone'},
    'month': set('jan feb mar apr may jun jul aug sep oct nov dec'.split()),
    'day_of_week': set('mon tue wed thu fri'.split())}
assert not raw.isna().any().any(), 'Unexpected nulls; report before changes'
for c, allowed in CATEGORICAL_DOMAINS.items():
    unexpected = set(raw[c]) - allowed
    assert not unexpected, f'{c}: unexpected categories {unexpected}'
    if pd.api.types.is_string_dtype(raw[c]):
        assert not raw[c].str.strip().eq('').any(), f'{c}: blank values require review'
ECONOMIC = ['emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']
COUNT_COLUMNS = ['age','duration','campaign','pdays','previous']
for c in COUNT_COLUMNS + ECONOMIC:
    assert pd.api.types.is_numeric_dtype(raw[c]), f'{c}: unexpected nonnumeric dtype'
    assert np.isfinite(raw[c]).all(), f'{c}: nonfinite values require review'
for c in COUNT_COLUMNS:
    assert raw[c].ge(0).all() and raw[c].mod(1).eq(0).all(), f'{c}: invalid count/day values'
assert raw.age.gt(0).all() and raw.campaign.ge(1).all()
quality_audit = pd.DataFrame({
    'dtype': raw.dtypes.astype(str), 'null_count': raw.isna().sum(),
    'unique_count': raw.nunique(dropna=False)})
display(quality_audit)
display(raw[COUNT_COLUMNS + ECONOMIC].agg(['min','max']).T)
print('All raw domain checks passed; no additional row deletion or value correction.')


All raw domain checks passed; no additional row deletion or value correction.


,dtype,null_count,unique_count
age,int64,0,78
job,object,0,12
marital,object,0,4
education,object,0,8
default,int64,0,3
housing,int64,0,3
loan,int64,0,3
contact,object,0,2
month,object,0,10
day_of_week,object,0,5


,min,max
age,17.000,98.000
duration,0.000,4918.000
campaign,1.000,56.000
pdays,0.000,999.000
previous,0.000,7.000
emp.var.rate,-3.400,1.400
cons.price.idx,92.201,94.767
cons.conf.idx,-50.800,-26.900
euribor3m,0.634,5.045
nr.employed,4963.600,5228.100


## 2. Remove exact original-row duplicates, keeping first

The duplicate key is **all 20 original columns, including y and the soon-to-be-excluded columns**. Row order is retained. Equality of the eventual modeling predictors is not a removal criterion. This implements the user's approval; without customer IDs we still cannot prove that identical rows represent the same person. No further observations are deleted.


In [3]:
duplicate_mask = raw.duplicated(keep='first')
duplicates_removed = int(duplicate_mask.sum())
assert duplicates_removed == 12, 'Duplicate count differs from approved snapshot; review'
dedup = raw.loc[~duplicate_mask].copy()
assert len(dedup) == 41176
assert dedup.index.equals(raw.index[~duplicate_mask])
assert not dedup.duplicated().any()
pd.testing.assert_frame_equal(dedup, raw.drop_duplicates(keep='first'), check_exact=True)
removed_target_counts = raw.loc[duplicate_mask, 'y'].value_counts().reindex([0,1], fill_value=0)
target_comparison = pd.DataFrame({
    'original_count': raw.y.value_counts().reindex([0,1], fill_value=0),
    'removed_count': removed_target_counts,
    'cleaned_count': dedup.y.value_counts().reindex([0,1], fill_value=0)})
target_comparison['original_percent'] = target_comparison.original_count / len(raw) * 100
target_comparison['cleaned_percent'] = target_comparison.cleaned_count / len(dedup) * 100
print('Removed:', duplicates_removed, '| remaining:', len(dedup))
display(target_comparison)


Removed: 12 | remaining: 41176


,original_count,removed_count,cleaned_count,original_percent,cleaned_percent
y,,,,,
0,36548,11,36537,88.734583,88.733728
1,4640,1,4639,11.265417,11.266272


## 3. Preserve explicit unknowns and exclude approved campaign-event fields

`job`, `marital` and `education` keep their exact text, including `unknown`. For `default`, `housing` and `loan`, a fixed dictionary replaces -1 with `unknown`, 0 with `no`, and 1 with `yes`. This is a human-readable category relabelling, **not Label/Ordinal/One-Hot/Target Encoding**. No frequencies or labels are used to learn the mapping. It prevents treating -1/0/1 as continuous values in the saved CSV; downstream numeric encoders/scalers remain out of scope.

Exclude `duration`, `campaign`, `contact`, `month`, `day_of_week`: these describe current-campaign calls or the recorded contact event, unavailable or unverified before campaign initiation. Raw columns are preserved in `dataset.csv`. No other feature-selection operation is performed.


In [4]:
EXCLUDED = ['duration','campaign','contact','month','day_of_week']
STATUS_COLUMNS = ['default','housing','loan']
TEXT_CATEGORIES = ['job','marital','education']
STATUS_LABELS = {-1:'unknown', 0:'no', 1:'yes'}
cleaned = dedup.drop(columns=EXCLUDED).copy()
for c in STATUS_COLUMNS:
    cleaned[c] = dedup[c].map(STATUS_LABELS).astype('string')
unknown_rows = []
for c in TEXT_CATEGORIES + STATUS_COLUMNS:
    unknown_code = 'unknown' if c in TEXT_CATEGORIES else -1
    before = int(raw[c].eq(unknown_code).sum())
    removed = int(raw.loc[duplicate_mask,c].eq(unknown_code).sum())
    after = int(cleaned[c].eq('unknown').sum())
    assert after == before - removed
    assert np.array_equal(cleaned[c].eq('unknown').to_numpy(), dedup[c].eq(unknown_code).to_numpy())
    unknown_rows.append({'feature':c,'original_unknown':before,'removed_with_duplicates':removed,
                         'cleaned_unknown':after,'original_percent':before/len(raw)*100,
                         'cleaned_percent':after/len(cleaned)*100})
unknown_comparison = pd.DataFrame(unknown_rows).set_index('feature')
display(unknown_comparison)
print('Status value sets:', {c: sorted(cleaned[c].unique()) for c in STATUS_COLUMNS})


Status value sets: {'default': ['no', 'unknown', 'yes'], 'housing': ['no', 'unknown', 'yes'], 'loan': ['no', 'unknown', 'yes']}


,original_unknown,removed_with_duplicates,cleaned_unknown,original_percent,cleaned_percent
feature,,,,,
job,330,0,330,0.801204,0.801438
marital,80,0,80,0.194231,0.194288
education,1731,1,1730,4.202680,4.201477
default,8597,1,8596,20.872584,20.876239
housing,990,0,990,2.403613,2.404313
loan,990,0,990,2.403613,2.404313


## 4. Deterministic sentinel-aware recency features

`recency_available = 1` when pdays is not 999, otherwise 0. `recency_days` keeps the actual value when available; otherwise it uses a fixed **0 placeholder**. Availability=0 does not mean “never contacted”. Availability=1 with days=0 is an actual zero-day recency, distinct from the placeholder. No statistical imputation or scaling is performed.

Keep `previous` unchanged. In particular, previous>0 with pdays=999 is not treated as erroneous. Drop raw pdays only from the derived modeling dataset. Historical scope and the reference date used to compute pdays still need confirmation: days measured at a later call are not necessarily days measured at campaign start.


In [5]:
sentinel = dedup.pdays.eq(999)
cleaned['recency_available'] = (~sentinel).astype('int64')
cleaned['recency_days'] = dedup.pdays.where(~sentinel, 0)
cleaned = cleaned.drop(columns=['pdays'])
# Retain y last, with no row reordering or index column exported.
FEATURES = ['age','job','marital','education','default','housing','loan','previous',
            'emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed',
            'recency_available','recency_days']
cleaned = cleaned[FEATURES + ['y']]

def history_counts(frame):
    return pd.Series({
        'pdays_999': int(frame.pdays.eq(999).sum()),
        'actual_recency': int(frame.pdays.ne(999).sum()),
        'previous_positive_pdays_999': int((frame.previous.gt(0) & frame.pdays.eq(999)).sum()),
        'actual_zero_day_recency': int(frame.pdays.eq(0).sum()),
        'previous_zero_actual_recency': int((frame.previous.eq(0) & frame.pdays.ne(999)).sum())})
history_comparison = pd.DataFrame({'original':history_counts(raw),'cleaned':history_counts(dedup)})
assert history_comparison.loc['previous_positive_pdays_999','cleaned'] == 4110
assert cleaned.loc[sentinel,'recency_available'].eq(0).all()
assert cleaned.loc[sentinel,'recency_days'].eq(0).all()
assert cleaned.loc[~sentinel,'recency_available'].eq(1).all()
pd.testing.assert_series_equal(cleaned.loc[~sentinel,'recency_days'], dedup.loc[~sentinel,'pdays'], check_names=False, check_exact=True)
assert cleaned.loc[dedup.pdays.eq(0),'recency_available'].eq(1).all()
pd.testing.assert_series_equal(cleaned.previous, dedup.previous, check_exact=True)
display(history_comparison)
case_counts = pd.DataFrame({'previous_positive':dedup.previous.gt(0),
                           'recency_available':cleaned.recency_available,
                           'recency_days':cleaned.recency_days})
display(case_counts.groupby(['previous_positive','recency_available']).agg(
    records=('recency_days','size'), minimum_days=('recency_days','min'), maximum_days=('recency_days','max')))


,original,cleaned
pdays_999,39673,39661
actual_recency,1515,1515
previous_positive_pdays_999,4110,4110
actual_zero_day_recency,15,15
previous_zero_actual_recency,0,0


records  minimum_days  maximum_days
previous_positive recency_available                                     
False             0                    35551             0             0
True              0                     4110             0             0
                  1                     1515             0            27

## 5. Validate the complete transformation contract

Assertions prove that the row set is exactly the approved keep-first set; unchanged quantitative and text fields equal their originals; status labels correspond exactly to original codes; recency values match the specified formulas; and only the approved columns are removed/replaced. This is stronger than merely checking column names for evidence of no scaling/encoding. No model-oriented preprocessing libraries are imported.

The modeling projection can contain repeated rows even after raw duplicates are removed. These are retained: identical predictor profiles or identical projected rows do not authorize further deletion. The target is separate from the predictor list and never used to engineer features.


In [6]:
validation = []
def require(condition, description):
    assert bool(condition), description
    validation.append({'check':description,'status':'PASS'})
require(cleaned.shape == (41176,16), 'Expected dimensions: 41,176 rows and 16 columns')
require(cleaned.index.equals(dedup.index), 'Only the 12 approved exact original-row duplicates removed')
require(cleaned.columns.tolist() == FEATURES + ['y'], 'Exactly 15 predictors plus y; no extra index/features')
require(not set(EXCLUDED + ['pdays']).intersection(cleaned.columns), 'Five approved exclusions and original pdays absent')
require(cleaned.y.isin([0,1]).all(), 'Target labels remain binary')
require(not cleaned.isna().any().any(), 'No unexpected nulls introduced')
require(cleaned.recency_available.isin([0,1]).all(), 'Recency availability remains a binary deterministic indicator')
require(cleaned.recency_days.ge(0).all(), 'Recency days nonnegative; sentinel is not elapsed time')
unchanged = [c for c in FEATURES + ['y'] if c not in STATUS_COLUMNS + ['recency_available','recency_days']]
for c in unchanged:
    pd.testing.assert_series_equal(cleaned[c], dedup[c], check_exact=True)
for c in STATUS_COLUMNS:
    pd.testing.assert_series_equal(cleaned[c], dedup[c].map(STATUS_LABELS).astype('string'), check_exact=True)
    require(not pd.api.types.is_numeric_dtype(cleaned[c]), f'{c}: human-readable categorical status, not continuous codes')
require(True, 'Unchanged features exactly equal original retained rows: no scaling, encoding, clipping or other transformations')
require(np.array_equal(cleaned.recency_available.to_numpy(), dedup.pdays.ne(999).astype('int64').to_numpy()), 'Recency availability formula matches every retained row')
require(np.array_equal(cleaned.recency_days.to_numpy(), np.where(dedup.pdays.eq(999),0,dedup.pdays)), 'Recency days formula matches every retained row')
pd.testing.assert_frame_equal(raw, raw_snapshot, check_exact=True)
require(hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_sha256, 'Original dataset bytes and in-memory values unchanged')
print('Projected exact duplicates retained:', int(cleaned.duplicated().sum()))
print('Predictor-only duplicates retained:', int(cleaned[FEATURES].duplicated().sum()))
display(pd.DataFrame(validation))
display(cleaned.dtypes.rename('cleaned_dtype').to_frame())
display(cleaned.head())


Projected exact duplicates retained: 5841
Predictor-only duplicates retained: 6488


,check,status
0,"Expected dimensions: 41,176 rows and 16 columns",PASS
1,Only the 12 approved exact original-row duplic...,PASS
2,Exactly 15 predictors plus y; no extra index/f...,PASS
3,Five approved exclusions and original pdays ab...,PASS
4,Target labels remain binary,PASS
5,No unexpected nulls introduced,PASS
6,Recency availability remains a binary determin...,PASS
7,Recency days nonnegative; sentinel is not elap...,PASS
8,"default: human-readable categorical status, no...",PASS
9,"housing: human-readable categorical status, no...",PASS


,cleaned_dtype
age,int64
job,object
marital,object
education,object
default,string[python]
housing,string[python]
loan,string[python]
previous,int64
emp.var.rate,float64
cons.price.idx,float64


,age,job,marital,education,default,housing,loan,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,recency_available,recency_days,y
0,56,housemaid,married,basic.4y,no,no,no,0,1.1,93.994,-36.4,4.857,5191.0,0,0,0
1,57,services,married,high.school,unknown,no,no,0,1.1,93.994,-36.4,4.857,5191.0,0,0,0
2,37,services,married,high.school,no,yes,no,0,1.1,93.994,-36.4,4.857,5191.0,0,0,0
3,40,admin.,married,basic.6y,no,no,no,0,1.1,93.994,-36.4,4.857,5191.0,0,0,0
4,56,services,married,high.school,no,no,yes,0,1.1,93.994,-36.4,4.857,5191.0,0,0,0


## 6. Availability review and readiness

The output follows the approved **campaign-start** feature schema but is not certified leakage-free or model-ready. The five event features are excluded; the following unresolved inputs are retained as requested rather than silently removed.

| Inputs | What is established | What needs confirmation before modeling |
|---|---|---|
| age, job, marital, education | Customer attributes can exist before a campaign | Verify campaign-start snapshots and no retrospective updates; divorced includes widowed |
| default, housing, loan | Historical financial status can exist before a campaign; unknowns explicitly retained | Confirm status recording/update dates and access before campaign initiation |
| previous | README describes prior-campaign counts | Verify cutoff excludes current/future contacts and consistent campaign scope |
| recency_available, recency_days from pdays | Deterministic representation of supplied historical recency; no assertion of never contacted | Confirm pdays reference date: an elapsed-day value calculated at last call may not be available at campaign start; obtain dates/snapshots or an approved alternative |
| emp.var.rate, nr.employed | Quarterly context indicators | Confirm campaign-start period and actual publication vintage; later quarter values/revisions are unsafe |
| cons.price.idx, cons.conf.idx | Monthly context indicators | Confirm values were published by campaign start, not tied to the later contact month |
| euribor3m | Daily context indicator | A last-call-day rate is not a campaign-start rate; confirm as-of date or obtain campaign-start values |
| y | Binary observed subscription outcome | Confirm outcome window and follow-up coverage align to customer/campaign priority task |

No campaign-start timestamps, full contact dates or customer identifiers are supplied. These metadata limitations prevent definitive temporal availability verification or client-disjoint validation. Do not use row order as verified chronology. Request as-of snapshots/release dates; obtain team approval for any later lagging, recomputation or exclusion. Strong socioeconomic correlations remain documented, not used for further feature selection.

**Next-phase recommendations, not implemented:** encode the six remaining categorical predictors (job, marital, education, default, housing, loan) later. LR will likely benefit from training-only scaling of age, previous, five economic indicators, and an approved conditional treatment of recency_days; keep recency_available unscaled. Decision Tree/Random Forest generally need no numerical scaling. Fit all learned preprocessing only on training folds after the planned stratified 80/20 split and stratified five-fold CV are approved. No split happens here; 41,188-based sample sizes in the old plan must be recalculated for 41,176 rows. Exact duplicate deletion does not remove repeated-client risk. Previously viewing full-data EDA also needs disclosure when describing a future holdout.

Legitimate extremes, rare levels, the 4,110 history records and all unknown observations remain. No additional filtering, category merging, inferred socioeconomic lags or imputation is authorized. Approval needed: feature availability and label window, campaign-start recency/economic definitions, later encoding/scaling choices, and validation design limits. The approved cleaning operations are complete; predictive readiness remains conditional on this review.


## 7. Export and independently verify the CSV

Outputs preserve human-readable categories and original units. Only the two named derived artifacts are overwritten on rerun, never `dataset.csv`. Export is followed by a read-back check of schema, row count, labels, numerical values and categories. Runtime tables and summary are computed from the current validated input rather than invented results.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaned_path = OUTPUT_DIR / '02_cleaned_dataset.csv'
summary_path = OUTPUT_DIR / '02_data_cleaning_summary.md'
assert cleaned_path.resolve() != DATA_PATH and summary_path.resolve() != DATA_PATH
cleaned.to_csv(cleaned_path, index=False)
loaded = pd.read_csv(cleaned_path, dtype={c:'string' for c in STATUS_COLUMNS}, float_precision='round_trip')
pd.testing.assert_frame_equal(loaded, cleaned.reset_index(drop=True), check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_sha256
print('CSV export/read-back passed:', cleaned_path)

def markdown_table(frame):
    # Avoid optional tabulate dependency in Colab/local execution.
    t = frame.reset_index()
    def render(v):
        if isinstance(v, (float, np.floating)):
            return f'{v:.6f}'
        return str(v).replace('|',' / ').replace('\n',' ')
    lines = ['| ' + ' | '.join(map(str,t.columns)) + ' |',
             '| ' + ' | '.join(['---']*len(t.columns)) + ' |']
    lines += ['| ' + ' | '.join(render(v) for v in row) + ' |' for row in t.itertuples(index=False,name=None)]
    return '\n'.join(lines)
summary = f"""# Phase 2B — Data cleaning summary

## Scope and implemented decisions

Scenario: predict subscription before the current marketing campaign begins. Implemented only the user's approved deterministic cleaning; Phase 2B supersedes Phase 2A's tentative pre-call scenario and duplicate-retention proposal. Reviewed README, project DOCX, Phase 1 notebook/summary, Phase 2A review/table and raw CSV. Source CSV SHA-256: `{source_sha256}`.

Original dimensions: **{len(raw):,} × {len(raw.columns)}** (19 predictors plus target). Cleaned dimensions: **{len(cleaned):,} × {len(cleaned.columns)}** (15 predictors plus target). The README headline says 21 columns, but its detailed list and supplied data both have 20; no missing feature has been invented.

## Approved exclusions

- duration: last-call duration is unavailable before the call/campaign.
- campaign: current-campaign contacts are not known before campaign initiation.
- contact, month, day_of_week: recorded contact-event channel/date are not established campaign-start inputs.
- Original pdays: replaced by the two approved derived recency fields, not otherwise feature-selected away.

All original columns and records remain unchanged in dataset.csv.

## Duplicate removal and target distribution

Exactly **{duplicates_removed}** duplicate rows beyond first occurrences were removed using **all 20 original columns before exclusions**. Kept first occurrences in original order; no other rows deleted. Retained row count: {len(dedup):,}. Customer identity still cannot be verified without IDs. Projected duplicate rows ({int(cleaned.duplicated().sum())}) and repeated predictor profiles ({int(cleaned[FEATURES].duplicated().sum())} beyond first) were retained and are not an additional deletion criterion.

{markdown_table(target_comparison.rename_axis('y'))}

Target counts change only by approved duplicate removal: 11 no and 1 yes removed. Class proportions change slightly, not identically. Only 0/1 target labels exist; no target imputation or extra exclusion occurred.

## Unknown-value handling

job, marital and education retain their exact text including unknown. default/housing/loan use the deterministic human-readable mapping -1 → unknown, 0 → no, 1 → yes. This is category relabelling for clarity, not model-oriented categorical encoding or a learned transformation. CSV status columns read as strings; do not reinterpret them as continuous values.

{markdown_table(unknown_comparison)}

Education and default unknown counts each fall by one only because a removed exact duplicate carried that unknown. All other unknown counts are unchanged; retained-row unknown masks match raw retained rows exactly. No unknown observations were dropped or statistically imputed.

## Pdays transformation and verification

- recency_available = 1 iff original pdays != 999; otherwise 0.
- recency_days = original pdays when available, otherwise fixed placeholder 0.
- previous is unchanged; no inference that mixed history records are erroneous.
- Availability=0 with days=0 is an unavailable-recency placeholder, not an actual zero-day interval. All {int(dedup.pdays.eq(0).sum())} actual zero-day cases have availability=1.

{markdown_table(history_comparison.rename_axis('case'))}

All 4,110 previous>0/sentinel records remain with previous unchanged, availability=0 and days=0. Actual recency range: {int(dedup.loc[~sentinel,'pdays'].min())}–{int(dedup.loc[~sentinel,'pdays'].max())} days. No invalid/nonfinite/negative day values were found. The sentinel count falls by 12 solely through duplicate removal. Availability does not assert whether a person was ever contacted. No scaling or statistical recency imputation occurred.

## Remaining feature list and types

Predictors (in export order): {', '.join(FEATURES)}.
Target: y (binary 0/1, not a predictor).

Categorical: job, marital, education, default, housing, loan. Quantitative: age, previous, emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed, recency_days. Deterministic binary indicator: recency_available.

## Observed data quality and validation evidence

No raw/cleaned nulls, blank categorical strings, undocumented category values, nonfinite numerical values, negative contact/recency counts or invalid target labels were found. Numeric/code domains were checked before transformations. Legitimate extremes remain: age {int(cleaned.age.min())}–{int(cleaned.age.max())}, previous {int(cleaned.previous.min())}–{int(cleaned.previous.max())}; negative economic rates/index values were preserved. Rare groups remain (default=yes {int(cleaned.default.eq('yes').sum())}; education=illiterate {int(cleaned.education.eq('illiterate').sum())}). No outlier correction or feature selection beyond the approved exclusions was applied.

Notebook assertions check raw duplicate count, exact keep-first row indices, expected output dimensions/columns, unknown masks/count conservation, unchanged columns exactly equal in value/dtype, exact status relabelling, both recency formulas row by row, valid labels, no introduced nulls, unchanged raw-file hash and exact exported CSV read-back. All {len(validation)} recorded validation checks passed, together with the detailed equality assertions and CSV read-back check. No estimator/encoder/scaler was fitted or applied. Original in-memory values and CSV bytes are unchanged.

## Unresolved questions: campaign-start availability

The output is a cleaned candidate modeling dataset, **not certified leakage-free**. All remaining fields were retained as approved; collection timing cannot be established from this CSV alone.

- Customer attributes and financial statuses: require campaign-start snapshots, not retrospective changes after subscription.
- previous: verify prior-campaign cutoff excludes current/future events and matches the intended campaign window.
- pdays-derived recency: confirm its reference date. A value calculated at a later contact date is not automatically valid at campaign start; obtain start dates/as-of history before using it predictively. Confirm the meaning/scope of the 4,110 mixed history records without marking them erroneous.
- emp.var.rate/nr.employed (quarterly), cons.price.idx/cons.conf.idx (monthly), euribor3m (daily): confirm publication dates, revision vintage and campaign-start alignment. Last-contact-period economic values may be unavailable at campaign start. No lag or replacement has been invented.
- y: confirm outcome window and follow-up completeness for campaign-start customer prioritization.
- No customer ID, campaign-start timestamp or full dated history exists here. Repeated-client and temporal leakage cannot be excluded; row order is not verified chronology.

Recommendations for discussion: obtain timestamped customer/history snapshots, campaign-start dates and economic publication vintages. Seek team approval for any later recomputation, lagging or additional feature exclusion. Strong socioeconomic correlations persist; no automatic selection was made. No psychographic feature is inferred from call timing.

## Readiness and subsequent-phase boundaries

Approved cleaning is complete and reproducible. Do not train until timing/availability concerns are resolved or an explicit limited-use assumption is approved. Later encode the six categorical predictors; for LR consider training-only scaling of age, previous, economic indicators and an approved conditional recency treatment, keeping recency_available unscaled. Trees/RF ordinarily need no numeric scaling. Exact encoding/scaling methods require the next phase's approval.

No scaling, One-Hot/Label/Ordinal/Target Encoding, log transform, train-test split, cross-validation, SMOTE/undersampling or model training was performed. Only approved category relabelling and deterministic features were applied. Planned next steps are stratified 80/20 and stratified five-fold CV; update counts for 41,176 records rather than old 41,188-based estimates. All learned preprocessing must be fitted on training folds only. Disclose prior whole-dataset EDA when discussing the eventual held-out set. Duplicate deletion alone does not establish client/time-disjoint evaluation.

Deliverables: 02_data_cleaning.ipynb, 02_cleaned_dataset.csv, 02_data_cleaning_summary.md, all in the repository. Team approval is still needed for unresolved feature availability, historical/label definitions, any additional changes, and later encoding/scaling/validation choices. No subsequent phase was started.
"""
summary_path.write_text(summary, encoding='utf-8')
print('Summary saved:', summary_path)
print('Completed approved cleaning only. No split, scaling, encoding or model training.')


CSV export/read-back passed: /workspace/bank-marketing-ml/02_cleaned_dataset.csv
Summary saved: /workspace/bank-marketing-ml/02_data_cleaning_summary.md
Completed approved cleaning only. No split, scaling, encoding or model training.
